# 🧩 Protein Domains and Pfam

---

## Learning Objectives

- Understand protein domain concepts
- Work with Pfam and InterPro
- Learn Hidden Markov Model basics
- Analyze domain architecture

---

## What are Protein Domains?

```
┌─────────────────────────────────────────────────────────────────────────┐
│                      PROTEIN DOMAINS                                    │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Domain: Independent structural/functional unit within a protein       │
│                                                                         │
│   Multi-domain protein example (Src kinase):                            │
│                                                                         │
│   N────┬────────┬────────────────────────────────┬─────C                │
│        │  SH3   │         SH2          │ Kinase │                       │
│        │ domain │        domain        │ domain │                       │
│        └────────┴──────────────────────┴────────┘                       │
│         50-100aa    ~100aa              ~250aa                          │
│                                                                         │
│   Properties of domains:                                                │
│   • Fold independently                                                  │
│   • Often have distinct function                                        │
│   • Found in multiple proteins (domain shuffling)                       │
│   • Evolutionarily conserved                                            │
│                                                                         │
│   Domain databases:                                                     │
│   • Pfam - protein families and domains                                 │
│   • InterPro - integrated domain annotations                            │
│   • SMART - signaling domain architecture                               │
│   • CDD - Conserved Domain Database (NCBI)                              │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---

## Hidden Markov Models (HMM)

```
┌─────────────────────────────────────────────────────────────────────────┐
│                   PROFILE HMM FOR DOMAINS                               │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Profile HMM architecture:                                             │
│                                                                         │
│   Begin → M₁ → M₂ → M₃ → M₄ → ... → Mₙ → End                            │
│            ↓    ↓    ↓    ↓          ↓                                  │
│           I₁   I₂   I₃   I₄         Iₙ  (Insert states)                 │
│            ↑    ↑    ↑    ↑          ↑                                  │
│           D₁   D₂   D₃   D₄         Dₙ  (Delete states)                 │
│                                                                         │
│   Match states (M): Emit amino acid with position-specific probs        │
│   Insert states (I): Handle insertions (any amino acid)                 │
│   Delete states (D): Handle deletions (no emission)                     │
│                                                                         │
│   Transitions:                                                          │
│   M→M: Continue in alignment                                            │
│   M→I: Insertion in query                                               │
│   M→D: Deletion in query                                                │
│   I→I: Extended insertion                                               │
│   I→M: End insertion                                                    │
│                                                                         │
│   E-value: Expected number of false positives at this score             │
│   • E < 0.01: Significant hit (likely true domain)                      │
│   • E > 1: Not significant (likely false positive)                      │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
class SimpleDomainHMM:
    """
    Simplified domain HMM for demonstration.
    
    Real HMMs use HMMER (hmmer.org).
    """
    
    def __init__(self, name, length):
        self.name = name
        self.length = length
        
        # Emission probabilities (simplified: just conserved positions)
        # In reality, each position has 20 AA emission probs
        self.conserved_pattern = None
        
    def build_from_alignment(self, sequences):
        """
        Build simple profile from alignment.
        
        Stores most common residue at each position.
        """
        if not sequences:
            return
        
        # Find consensus
        self.length = len(sequences[0])
        pattern = []
        
        for pos in range(self.length):
            residues = [seq[pos] for seq in sequences if seq[pos] != '-']
            if residues:
                # Most common residue
                from collections import Counter
                counts = Counter(residues)
                most_common = counts.most_common(1)[0]
                conservation = most_common[1] / len(residues)
                
                if conservation > 0.5:
                    pattern.append((most_common[0], conservation))
                else:
                    pattern.append(('X', conservation))  # Variable
            else:
                pattern.append(('-', 0))  # Gap
        
        self.conserved_pattern = pattern
    
    def score_sequence(self, sequence):
        """
        Score a sequence against the profile.
        
        Returns simple match score.
        """
        if not self.conserved_pattern:
            return 0
        
        score = 0
        for i, (cons_aa, cons_score) in enumerate(self.conserved_pattern):
            if i < len(sequence):
                if cons_aa == 'X':  # Variable position
                    score += 0.5
                elif cons_aa == sequence[i]:
                    score += cons_score * 2  # Reward match
                else:
                    score -= 0.5  # Penalize mismatch at conserved
        
        return score

# Example: SH2 domain-like pattern
sh2_alignment = [
    "FLVRESETT",
    "YLVRESETA",
    "FLVRESQST",
    "WLVKESETT",
    "FLVRESETT"
]

hmm = SimpleDomainHMM("SH2_like", 9)
hmm.build_from_alignment(sh2_alignment)

print(f"Domain: {hmm.name}")
print(f"\nConserved pattern:")
for i, (aa, cons) in enumerate(hmm.conserved_pattern):
    bar = '█' * int(cons * 10)
    print(f"  Position {i+1}: {aa} ({cons:.0%}) {bar}")

---

## Pfam Database

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        PFAM DATABASE                                    │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Pfam entry types:                                                     │
│                                                                         │
│   Family (Pfam-A): Curated, HMM profiles                                │
│   • High-quality alignments                                             │
│   • Expert annotation                                                   │
│   • ~20,000 families                                                    │
│                                                                         │
│   Clan: Group of related families                                       │
│   • Share common evolutionary origin                                    │
│   • May have similar structure/function                                 │
│                                                                         │
│   Entry example (PF00069 - Protein kinase):                             │
│   ┌────────────────────────────────────────────┐                        │
│   │ Accession: PF00069                         │                        │
│   │ Name: Pkinase                              │                        │
│   │ Type: Domain                               │                        │
│   │ Description: Protein kinase domain         │                        │
│   │ Clan: CL0016 (Protein kinase superfamily)  │                        │
│   │ Sequences: >500,000                        │                        │
│   │ Average length: ~260 aa                    │                        │
│   │ HMM length: 264 match states               │                        │
│   └────────────────────────────────────────────┘                        │
│                                                                         │
│   Common Pfam domains in human proteome:                                │
│   • Zinc finger (C2H2) - PF00096                                        │
│   • Immunoglobulin - PF00047                                            │
│   • Protein kinase - PF00069                                            │
│   • WD40 repeat - PF00400                                               │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
def parse_pfam_output(pfam_text):
    """
    Parse HMMER/Pfam domain scan output (tblout format).
    
    Returns list of domain hits.
    """
    hits = []
    
    for line in pfam_text.strip().split('\n'):
        if line.startswith('#'):
            continue
        
        fields = line.split()
        if len(fields) >= 19:
            hit = {
                'target_name': fields[0],
                'target_accession': fields[1],
                'query_name': fields[2],
                'query_accession': fields[3],
                'e_value': float(fields[4]),
                'score': float(fields[5]),
                'bias': float(fields[6]),
                'domain_e_value': float(fields[7]),
                'domain_score': float(fields[8]),
                'domain_bias': float(fields[9]),
                'ali_from': int(fields[17]),
                'ali_to': int(fields[18]),
                'description': ' '.join(fields[22:]) if len(fields) > 22 else ''
            }
            hits.append(hit)
    
    return hits

def visualize_domain_architecture(protein_length, domains):
    """
    Create ASCII visualization of domain architecture.
    """
    scale = 60  # Characters for full protein
    
    # Draw protein backbone
    print(f"\nDomain Architecture ({protein_length} aa):")
    print("N─" + "─" * scale + "─C")
    
    # Draw domains
    architecture = list('─' * scale)
    
    for domain in sorted(domains, key=lambda x: x.get('start', 0)):
        start = domain.get('start', domain.get('ali_from', 0))
        end = domain.get('end', domain.get('ali_to', 100))
        name = domain.get('name', domain.get('target_name', '?'))[:8]
        
        # Scale positions
        s = int(start / protein_length * scale)
        e = int(end / protein_length * scale)
        
        # Draw domain box
        for i in range(s, min(e, scale)):
            architecture[i] = '█'
        
        # Print domain label
        mid = (s + e) // 2
        print(f"  {' ' * s}[{name}] ({start}-{end})")
    
    print("  " + ''.join(architecture))
    print(f"  0{' ' * (scale//2 - 2)}{protein_length//2}{' ' * (scale//2 - 4)}{protein_length}")

# Example: Src kinase domains
src_domains = [
    {'name': 'SH3', 'start': 84, 'end': 145},
    {'name': 'SH2', 'start': 151, 'end': 248},
    {'name': 'Pkinase', 'start': 270, 'end': 520}
]

visualize_domain_architecture(536, src_domains)

---

## InterPro Integration

```
┌─────────────────────────────────────────────────────────────────────────┐
│                      INTERPRO DATABASE                                  │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   InterPro integrates predictions from multiple databases:              │
│                                                                         │
│   ┌─────────────┬─────────────────────────────────────┐                 │
│   │ Database    │ Focus                               │                 │
│   ├─────────────┼─────────────────────────────────────┤                 │
│   │ Pfam        │ Protein families (HMM)              │                 │
│   │ PROSITE     │ Patterns and profiles               │                 │
│   │ PRINTS      │ Fingerprints (motif groups)         │                 │
│   │ SMART       │ Signaling domains                   │                 │
│   │ PANTHER     │ Protein families + GO               │                 │
│   │ CDD         │ Conserved domains (NCBI)            │                 │
│   │ SUPERFAMILY │ Structural superfamilies            │                 │
│   │ Gene3D      │ CATH structural domains             │                 │
│   └─────────────┴─────────────────────────────────────┘                 │
│                                                                         │
│   Benefits of InterPro:                                                 │
│   • Single query searches all databases                                 │
│   • Non-redundant domain assignments                                    │
│   • Linked to GO terms                                                  │
│   • Consistent identifiers (IPR######)                                  │
│                                                                         │
│   InterProScan workflow:                                                │
│   Sequence → [Pfam] [PROSITE] [SMART] ... → Merge → InterPro entry      │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Common domain accessions and their functions
DOMAIN_INFO = {
    'PF00069': {
        'name': 'Protein kinase',
        'interpro': 'IPR000719',
        'function': 'Phosphorylation of substrates',
        'common_in': 'Signaling proteins'
    },
    'PF00017': {
        'name': 'SH2 domain',
        'interpro': 'IPR000980',
        'function': 'Binds phosphotyrosine',
        'common_in': 'Signal transduction'
    },
    'PF00018': {
        'name': 'SH3 domain',
        'interpro': 'IPR001452',
        'function': 'Binds proline-rich sequences',
        'common_in': 'Signaling proteins'
    },
    'PF00096': {
        'name': 'Zinc finger C2H2',
        'interpro': 'IPR007087',
        'function': 'DNA binding',
        'common_in': 'Transcription factors'
    },
    'PF00400': {
        'name': 'WD40 repeat',
        'interpro': 'IPR001680',
        'function': 'Protein-protein interaction scaffold',
        'common_in': 'Signaling, chromatin'
    }
}

def lookup_domain(pfam_acc):
    """Look up domain information."""
    info = DOMAIN_INFO.get(pfam_acc)
    if info:
        print(f"\n{pfam_acc}: {info['name']}")
        print(f"  InterPro: {info['interpro']}")
        print(f"  Function: {info['function']}")
        print(f"  Common in: {info['common_in']}")
    else:
        print(f"\n{pfam_acc}: Not in local database")

# Look up some domains
for acc in ['PF00069', 'PF00017', 'PF00018']:
    lookup_domain(acc)

---

## 🏋️ Exercises

### Exercise 1: Domain Count
Count domain occurrences in a set of proteins.

### Exercise 2: Architecture Search
Find proteins with a specific domain combination.

### Exercise 3: Domain Enrichment
Identify overrepresented domains in a protein set.

---

## 📚 Resources

- [Pfam](https://www.ebi.ac.uk/interpro/entry/pfam/) - Protein families
- [InterPro](https://www.ebi.ac.uk/interpro/) - Integrated domains
- [HMMER](http://hmmer.org/) - HMM search tool
- [InterProScan](https://www.ebi.ac.uk/interpro/search/sequence/) - Online domain search